# CarsiDock 虚拟筛选教程

**论文引用:**
> Cai, H., et al. "CarsiDock: a deep learning paradigm for accurate protein-ligand docking and screening based on large-scale pre-training." *Chemical Science*, 2024.

## 核心创新

CarsiDock 是一种基于 **距离矩阵预测 + 构象搜索** 的分子对接方法。与 TankBind 共享相同的神经网络架构（预测蛋白-配体距离矩阵），但在 **后处理阶段** 有关键创新：

| 方法 | 坐标恢复策略 | 优势 | 劣势 |
|------|-------------|------|------|
| TankBind | 梯度下降拟合距离矩阵 | 可精确拟合任意距离矩阵 | 恢复的坐标可能化学不合理（键长/键角异常） |
| **CarsiDock** | **RDKit 构象搜索（类似 Vina）** | **保证键长键角化学合理性** | 搜索空间受限于生成构象的多样性 |

CarsiDock 的构象搜索策略模拟了传统对接软件（如 AutoDock Vina）的启发式搜索方式：
1. 生成多个候选构象
2. 在 **平移(3 DOF) + 旋转(3 DOF) + 扭转角(N DOF)** 的搜索空间中全局优化
3. 逐一打分并选择最优解

## 学习目标

通过本教程，你将学会：
1. 如何构建蛋白-配体 **距离矩阵预测** 模型（三角更新模块）
2. 如何使用 **RDKit 构象搜索** 从距离矩阵恢复配体 3D 坐标
3. 如何与 **梯度下降方法**（TankBind 基线）进行对比
4. 理解 **变换顺序（扭转 -> 旋转 -> 平移）** 的物理意义

In [ ]:
# ---- 项目根目录定位与路径设置 ----

def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

# ---- 导入依赖 ----

import os
import copy
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem
from scipy.spatial import distance_matrix
from scipy.stats import pearsonr
from scipy.optimize import differential_evolution
from collections import deque
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')

# ---- 版本信息 ----

version_info = {
    'Python 包': ['torch', 'rdkit', 'numpy', 'scipy', 'pandas', 'matplotlib'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        __import__('scipy').__version__,
        pd.__version__,
        __import__('matplotlib').__version__,
    ]
}
display(pd.DataFrame(version_info))

## 1. 超参数设置

以下超参数控制模型的特征维度、训练过程和构象搜索行为。

**关于构象搜索参数的说明：**
- `N_CONFORMERS`: RDKit 生成的初始构象数量，越多则搜索空间越大
- `DE_TOP_K`: 预筛选后保留的最优构象数，只对这些运行耗时的全局优化
- `DE_MAXITER` / `DE_POPSIZE`: differential_evolution 优化器的设置

In [ ]:
# ---------- 超参数 ----------
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长结构，逐样本处理
SEED = 42
MAX_DIST = 20.0             # 距离矩阵预测上限 (Angstrom)
POCKET_CUTOFF = 8.0         # 用于截取口袋原子的距离阈值 (Angstrom)
N_TRIANGLE_LAYERS = 3       # 三角更新层数
N_CONFORMERS = 50           # 构象搜索生成的构象数量
COORD_RECOVERY_STEPS = 2000 # 梯度下降坐标恢复步数（用于对比基线）
COORD_RECOVERY_LR = 0.1    # 梯度下降学习率
DE_MAXITER = 30             # differential_evolution 最大迭代数
DE_POPSIZE = 5              # differential_evolution 种群大小
DE_TOP_K = 5                # 预筛选后保留的最优构象数量（仅对这些运行 DE）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

display(pd.DataFrame({
    '超参数': [
        'ATOM_FEAT_DIM', 'HIDDEN_DIM', 'N_EPOCHS', 'LR', 'BATCH_SIZE',
        'MAX_DIST', 'POCKET_CUTOFF', 'N_TRIANGLE_LAYERS',
        'N_CONFORMERS', 'DE_MAXITER', 'DE_POPSIZE', 'DE_TOP_K',
        'COORD_RECOVERY_STEPS', 'COORD_RECOVERY_LR', 'DEVICE'
    ],
    '值': [
        ATOM_FEAT_DIM, HIDDEN_DIM, N_EPOCHS, LR, BATCH_SIZE,
        MAX_DIST, POCKET_CUTOFF, N_TRIANGLE_LAYERS,
        N_CONFORMERS, DE_MAXITER, DE_POPSIZE, DE_TOP_K,
        COORD_RECOVERY_STEPS, COORD_RECOVERY_LR, str(DEVICE)
    ],
    '说明': [
        '简化原子特征维度 (9 元素 one-hot + 1 芳香性)',
        '隐层维度',
        '训练轮数',
        '学习率',
        '变长结构，逐样本处理',
        '距离矩阵预测上限 (A)',
        '截取口袋原子的距离阈值 (A)',
        '三角更新层数（类似 AlphaFold2）',
        'RDKit 构象搜索生成的构象数',
        'differential_evolution 最大迭代数',
        'differential_evolution 种群大小',
        '预筛选后保留的最优构象数',
        '梯度下降坐标恢复步数（TankBind 基线）',
        '梯度下降学习率',
        '计算设备'
    ]
}))

## 2. 数据加载与特征提取

本节定义以下核心函数：

1. **构象搜索工具函数**（移植自 DiffDock）：
   - `axis_angle_to_matrix()`: Rodrigues 公式，轴角向量 -> 旋转矩阵
   - `get_rotatable_bonds()`: 获取配体的可旋转键列表
   - `modify_torsion_angles()`: 修改配体扭转角（BFS + Rodrigues 旋转）

2. **特征提取函数**：
   - `parse_coreset()`: 解析 CoreSet.dat 获取 PDB ID 列表
   - `atom_features()`: 构建 10 维原子特征（元素 one-hot + 芳香性）
   - `load_mol()`: 用 RDKit 加载分子文件
   - `compute_rmsd()`: 计算 RMSD
   - `build_carsidock_data()`: 为单个复合物构建完整数据

In [ ]:
# ============================================================
# 构象搜索工具函数（移植自 toy_diffdock.py）
# ============================================================

def axis_angle_to_matrix(axis_angle):
    """
    Rodrigues 公式: 轴角向量 -> 旋转矩阵

    输入: axis_angle (3,) numpy 数组，方向为旋转轴，模长为旋转角度（弧度）
    输出: (3, 3) 旋转矩阵

    公式: R = I + sin(theta)*K + (1-cos(theta))*K^2
    其中 K 是旋转轴 k 的反对称矩阵:
        K = [[  0, -k3,  k2],
             [ k3,   0, -k1],
             [-k2,  k1,   0]]
    """
    axis_angle = np.asarray(axis_angle, dtype=np.float64)
    angle = np.linalg.norm(axis_angle)
    if angle < 1e-8:
        return np.eye(3, dtype=np.float32)
    k = axis_angle / angle                  # 单位旋转轴
    K = np.array([[0, -k[2], k[1]],         # 反对称矩阵
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    return R.astype(np.float32)


def get_rotatable_bonds(mol):
    """
    获取配体的可旋转键列表（非环单键，不含氢原子）

    可旋转键定义:
      - 单键（非双键、三键、芳香键）
      - 不在环中（环内键旋转会破坏环结构）
      - 两端原子都不是氢（氢原子的旋转无物理意义）
      - 两端原子的度数 > 1（末端原子旋转无意义）
    """
    rot_bonds = []
    for bond in mol.GetBonds():
        if (bond.GetBondType() == Chem.rdchem.BondType.SINGLE
                and not bond.IsInRing()
                and bond.GetBeginAtom().GetAtomicNum() != 1
                and bond.GetEndAtom().GetAtomicNum() != 1
                and bond.GetBeginAtom().GetDegree() > 1
                and bond.GetEndAtom().GetDegree() > 1):
            rot_bonds.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
    return rot_bonds


def modify_torsion_angles(coords, mol, rot_bonds, delta_angles):
    """
    修改配体扭转角: 对每个可旋转键，用 BFS 确定旋转侧，再用 Rodrigues 公式旋转

    算法:
      对于每个可旋转键 (u, v) 和对应的旋转角度 delta_theta:
        1. 从 v 出发做 BFS（不经过 u），找到 v 侧的所有原子
        2. 旋转轴方向 = u -> v
        3. 用 Rodrigues 公式将 v 侧原子绕旋转轴旋转 delta_theta（以 u 为支点）

    这种 BFS 方法保证了只旋转键一侧的原子，不影响另一侧。
    """
    new_coords = coords.copy()
    for (u, v), angle in zip(rot_bonds, delta_angles):
        if abs(angle) < 1e-8:
            continue
        # BFS 从 v 出发，不经过 u，找到 v 侧的所有原子
        visited = set()
        queue = deque([v])
        visited.add(v)
        while queue:
            node = queue.popleft()
            for neighbor in [n.GetIdx() for n in mol.GetAtomWithIdx(node).GetNeighbors()]:
                if neighbor != u and neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        # 旋转轴: u -> v 方向
        axis = new_coords[v] - new_coords[u]
        axis_norm = np.linalg.norm(axis)
        if axis_norm < 1e-8:
            continue
        axis = axis / axis_norm
        # 用 Rodrigues 公式旋转 v 侧原子（以 u 为支点）
        rot_mat = axis_angle_to_matrix(axis * angle)
        for idx in visited:
            new_coords[idx] = rot_mat @ (new_coords[idx] - new_coords[u]) + new_coords[u]
    return new_coords


# ============================================================
# 特征提取函数
# ============================================================

# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。
    注意：CarsiDock 是对接模型，logKa 仅用于获取有效 PDB ID 列表。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(labels)} 个复合物")
    return labels


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol


def compute_rmsd(pred_coords, true_coords):
    """计算两组坐标之间的 RMSD (均方根偏差)。
    pred_coords, true_coords: (N, 3) numpy 数组
    公式: RMSD = sqrt( mean( sum_i (pred_i - true_i)^2 ) )
    """
    diff = pred_coords - true_coords
    return np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))


def build_carsidock_data(pdbid):
    """
    为单个蛋白-配体复合物构建 CarsiDock 数据。

    与 TankBind 的 build_data 基本相同，但额外返回配体 RDKit Mol 对象，
    以便后续构象搜索时使用 RDKit 生成构象。

    返回:
      prot_feats  : (N_p, 10)   蛋白口袋原子特征
      lig_feats   : (N_l, 10)   配体原子特征
      prot_coords : (N_p, 3)    蛋白口袋原子坐标
      lig_coords  : (N_l, 3)    配体原子坐标 (晶体结构真实坐标)
      dist_matrix : (N_p, N_l)  蛋白-配体真实距离矩阵
      lig_mol     : RDKit Mol   去氢后的配体分子对象（用于构象生成）
    如果解析失败则返回 None。
    """
    pocket_path = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_pocket.pdb")
    ligand_mol2 = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    # ---- 5. 截取离配体较近的口袋原子，降低计算量 ----
    lig_center = lig_coords.mean(axis=0)
    prot_dists_to_center = np.linalg.norm(prot_coords - lig_center, axis=1)
    pocket_mask = prot_dists_to_center < POCKET_CUTOFF
    if pocket_mask.sum() < 3:
        top_k = min(20, len(prot_coords))
        pocket_idx = np.argsort(prot_dists_to_center)[:top_k]
        prot_coords = prot_coords[pocket_idx]
        prot_feats = prot_feats[pocket_idx]
    else:
        prot_coords = prot_coords[pocket_mask]
        prot_feats = prot_feats[pocket_mask]

    # ---- 6. 计算蛋白-配体真实距离矩阵 ----
    dist_mat = distance_matrix(prot_coords, lig_coords).astype(np.float32)
    dist_mat = np.clip(dist_mat, 0, MAX_DIST)

    return prot_feats, lig_feats, prot_coords, lig_coords, dist_mat, lig_mol


print("数据加载与特征提取函数定义完成。")

In [ ]:
# ---- 预处理所有复合物 ----
print("正在构建 CarsiDock 数据...")
labels = parse_coreset(str(CORESET_FILE))

all_data = []       # (prot_feats, lig_feats, prot_coords, lig_coords, dist_matrix)
all_mols = []       # RDKit Mol 对象
failed = 0
for pdbid in sorted(labels.keys()):
    result = build_carsidock_data(pdbid)
    if result is None:
        failed += 1
        continue
    prot_f, lig_f, prot_c, lig_c, dist_mat, lig_mol = result
    all_data.append((prot_f, lig_f, prot_c, lig_c, dist_mat))
    all_mols.append(lig_mol)

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# ---- 展示一个样本的数据形状 ----
sample = all_data[0]
display(pd.DataFrame({
    '数据': ['蛋白口袋特征', '配体特征', '蛋白口袋坐标', '配体坐标', '距离矩阵'],
    '形状': [str(sample[0].shape), str(sample[1].shape), str(sample[2].shape),
            str(sample[3].shape), str(sample[4].shape)],
    '含义': [
        f'{sample[0].shape[0]} 个口袋原子 x {sample[0].shape[1]} 维特征',
        f'{sample[1].shape[0]} 个配体原子 x {sample[1].shape[1]} 维特征',
        f'{sample[2].shape[0]} 个口袋原子 x 3D 坐标',
        f'{sample[3].shape[0]} 个配体原子 x 3D 坐标',
        f'{sample[4].shape[0]} 蛋白原子 x {sample[4].shape[1]} 配体原子 距离矩阵'
    ]
}))

## 3. 数据集与数据加载器

由于每个蛋白-配体复合物的原子数不同（变长结构），我们使用 `BATCH_SIZE=1` 逐样本处理。

**注意：** `lig_mol`（RDKit Mol 对象）单独存储，不经过 DataLoader 的 collate，因为 Python 对象不能直接转为 tensor。通过返回索引 `idx`，在需要时从 `mol_list` 取回对应的分子对象。

In [ ]:
class CarsiDockDataset(Dataset):
    """CarsiDock 数据集。每个样本包含蛋白/配体特征、坐标、真实距离矩阵。
    注意: lig_mol 单独存储，不经过 DataLoader 的 collate（Python 对象不能直接转 tensor）。"""
    def __init__(self, data_list, mol_list):
        # data_list: list of (prot_feats, lig_feats, prot_coords, lig_coords, dist_matrix)
        # mol_list: list of RDKit Mol
        self.data = data_list
        self.mols = mol_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prot_f, lig_f, prot_c, lig_c, dist_mat = self.data[idx]
        return (torch.FloatTensor(prot_f),       # (N_p, 10)
                torch.FloatTensor(lig_f),        # (N_l, 10)
                torch.FloatTensor(prot_c),       # (N_p, 3)
                torch.FloatTensor(lig_c),        # (N_l, 3)
                torch.FloatTensor(dist_mat),     # (N_p, N_l)
                idx)                             # 索引，用于从 mol_list 取分子


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
train_mols = [all_mols[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]
test_mols = [all_mols[i] for i in test_idx]

train_dataset = CarsiDockDataset(train_data, train_mols)
test_dataset = CarsiDockDataset(test_data, test_mols)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

display(pd.DataFrame({
    '数据集': ['训练集', '测试集'],
    '样本数': [len(train_data), len(test_data)],
    'Batch Size': [BATCH_SIZE, BATCH_SIZE]
}))

# ---- 展示一个 batch 的数据形状 ----
sample_batch = next(iter(train_loader))
print("\n单个 batch 各张量形状:")
names = ['蛋白特征', '配体特征', '蛋白坐标', '配体坐标', '距离矩阵', '索引']
for name, tensor in zip(names, sample_batch):
    if isinstance(tensor, torch.Tensor):
        print(f"  {name}: {tensor.shape}")
    else:
        print(f"  {name}: {tensor}")

## 4. 模型架构

ToyCarsiDock 的网络架构与 TankBind 完全相同：

```
蛋白原子特征 (N_p, 10) --- Linear ---+
                                      +-- 外积 (N_p, N_l, hidden) -- 三角更新 x3 -- 距离预测头 -- (N_p, N_l)
配体原子特征 (N_l, 10) --- Linear ---+
```

**三角更新模块 (TriangleUpdate):**
- 灵感来自 AlphaFold2 的三角乘法更新
- 核心思想：蛋白原子 i 到配体原子 j 的距离表示 z_ij，应当与经过中间节点 k 的路径 (z_ik, z_kj) 保持几何一致性
- 更新规则: `z_ij += Linear(sigmoid(left_proj(z)) * sigmoid(right_proj(z)))`

**CarsiDock 的核心创新不在网络架构，而在后处理：**
- TankBind: 用梯度下降从距离矩阵恢复坐标（可能产生化学不合理的构象）
- CarsiDock: 用 RDKit 构象搜索，保证配体键长键角的化学合理性

In [ ]:
class TriangleUpdate(nn.Module):
    """三角更新模块（简化版）

    灵感来自 AlphaFold2 的三角乘法更新 (triangle multiplicative update)。
    核心思想：蛋白原子 i 到配体原子 j 的距离表示 z_ij，
    应当与经过中间节点 k 的路径 (z_ik, z_kj) 保持几何一致性。

    更新规则:
      z_ij += Linear(Sigma_k z_ik * z_kj)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.left_proj = nn.Linear(hidden_dim, hidden_dim)
        self.right_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, z):
        """
        参数:
          z : (N_p, N_l, hidden_dim)  蛋白-配体对的隐表示矩阵

        返回:
          z_updated : (N_p, N_l, hidden_dim)  更新后的表示
        """
        z_normed = self.layer_norm(z)
        left = torch.sigmoid(self.left_proj(z_normed))
        right = torch.sigmoid(self.right_proj(z_normed))
        tri = left * right          # 逐元素门控 (适用于非方阵 N_p != N_l)
        return z + self.out_proj(tri)


class ToyCarsiDock(nn.Module):
    """
    CarsiDock 玩具模型 -- 基于距离矩阵预测 + 构象搜索的分子对接。

    模型架构与 TankBind 完全相同：
      1. 蛋白和配体原子各自通过线性嵌入层映射到隐空间；
      2. 外积构建蛋白-配体对的表示矩阵 z_ij = prot_h_i + lig_h_j；
      3. 多层三角更新增强几何一致性；
      4. 距离预测头将隐表示映射为距离值。

    CarsiDock 的核心创新不在网络架构，而在 **后处理**：
      - TankBind: 用梯度下降从距离矩阵恢复坐标（可能产生化学不合理的构象）
      - CarsiDock: 用 RDKit 构象搜索，保证配体键长键角的化学合理性
                   这更类似传统对接软件 (Vina) 的搜索策略
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM,
                 n_layers=N_TRIANGLE_LAYERS, max_dist=MAX_DIST):
        super().__init__()
        self.max_dist = max_dist

        # 1. 蛋白/配体原子嵌入层
        self.prot_embed = nn.Linear(atom_dim, hidden_dim)
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # 2. 多层三角更新
        self.triangle_layers = nn.ModuleList([
            TriangleUpdate(hidden_dim) for _ in range(n_layers)
        ])

        # 3. 距离预测头: hidden -> 1，输出经 Sigmoid 缩放到 [0, max_dist]
        self.dist_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, prot_feats, lig_feats):
        """
        参数:
          prot_feats : (N_p, atom_dim)  蛋白口袋原子特征
          lig_feats  : (N_l, atom_dim)  配体原子特征

        返回:
          pred_dist  : (N_p, N_l)       预测的蛋白-配体距离矩阵
        """
        prot_h = self.prot_embed(prot_feats)    # (N_p, hidden)
        lig_h = self.lig_embed(lig_feats)       # (N_l, hidden)

        # 构建 pair 表示矩阵
        z = prot_h.unsqueeze(1) + lig_h.unsqueeze(0)   # (N_p, N_l, hidden)

        # 三角更新
        for tri_layer in self.triangle_layers:
            z = tri_layer(z)

        # 距离预测
        pred_dist = self.dist_head(z).squeeze(-1) * self.max_dist  # (N_p, N_l)
        return pred_dist

In [ ]:
# ---- 坐标恢复方法 1: 构象搜索（CarsiDock 核心创新）----

def recover_coords_conformer_search(lig_mol, pred_dist, prot_coords,
                                    n_confs=N_CONFORMERS):
    """从预测距离矩阵恢复配体坐标——使用 Vina-like 构象搜索。

    核心思想（参考 CarsiDock 原始 docking_utils.py + conf_gen.py）：
      实现类似 AutoDock Vina 的搜索策略，在蛋白口袋内对配体进行
      平移(3 DOF) + 旋转(3 DOF) + 扭转角(N DOF) 的全局优化：
      1. 利用 RDKit 力场生成多个化学合理的配体初始构象；
      2. 获取配体可旋转键，定义 6+N 维搜索空间；
      3. 预筛选：用仅平移的快速打分选出 top-K 候选构象；
      4. 对 top-K 构象，用 differential_evolution 全局优化：
         - 扭转角修改内部自由度 -> 旋转施加刚体转动 -> 平移定位到口袋
         - 打分函数 = MSE(当前距离矩阵, 预测距离矩阵)
      5. 选择全局最优构象作为对接结果。

    变换顺序（与 Vina 一致）：扭转 -> 旋转 -> 平移
      - 扭转修改分子内部构象（改变二面角）
      - 旋转改变分子整体朝向（刚体旋转）
      - 平移将分子定位到口袋空间中（刚体平移）

    优势：
      - 生成的构象保证了键长、键角的化学合理性（由 RDKit 力场保证）
      - 通过旋转和扭转优化，搜索空间远大于仅平移的版本
      - differential_evolution 全局优化避免陷入局部最优

    参数:
      lig_mol    : RDKit Mol   配体分子对象（需带拓扑信息）
      pred_dist  : (N_p, N_l)  numpy 数组，预测的蛋白-配体距离矩阵
      prot_coords: (N_p, 3)    numpy 数组，蛋白原子坐标
      n_confs    : int         生成的构象数量

    返回:
      best_coords: (N_l, 3) numpy 数组，最优构象的坐标
    """
    n_lig = pred_dist.shape[1]
    pocket_center = prot_coords.mean(axis=0)    # (3,)

    # Step 1: 用 RDKit 生成多个 3D 构象
    # 参考 CarsiDock 的 single_conf_gen: 加氢 -> EmbedMultipleConfs -> 去氢
    mol = copy.deepcopy(lig_mol)
    mol = Chem.AddHs(mol)
    conf_ids = AllChem.EmbedMultipleConfs(
        mol, numConfs=n_confs, randomSeed=SEED,
        clearConfs=True, numThreads=0
    )
    mol = Chem.RemoveHs(mol)

    # 如果构象生成失败（极少数分子可能不兼容），回退到梯度下降
    if len(conf_ids) == 0:
        return recover_coords_gradient_descent(pred_dist, prot_coords, n_lig)

    # Step 2: 获取可旋转键，定义搜索空间维度
    rot_bonds = get_rotatable_bonds(mol)
    n_tor = len(rot_bonds)
    # 搜索空间: 平移(3) + 旋转(3) + 扭转(N)
    # - 平移: 口袋中心 +/- 5 A
    # - 旋转: 轴角表示，每个分量 [-pi, pi]
    # - 扭转: 每个可旋转键 [-pi, pi]
    bounds = ([(-5.0, 5.0)] * 3 +
              [(-np.pi, np.pi)] * 3 +
              [(-np.pi, np.pi)] * n_tor)

    # Step 3: 预筛选——用仅平移的快速打分选出 top-K 构象
    # 类似传统对接软件先粗筛再精搜的两阶段策略
    candidates = []     # (score, centered_coords)
    for conf_id in range(mol.GetNumConformers()):
        conf = mol.GetConformer(conf_id)
        init_coords = np.array([conf.GetAtomPosition(i)
                                for i in range(mol.GetNumAtoms())], dtype=np.float32)

        # 确保原子数匹配（去氢后应与特征提取时一致）
        if init_coords.shape[0] != n_lig:
            continue

        # 将构象中心化到原点（后续平移以 pocket_center 为基准）
        init_center = init_coords.mean(axis=0)
        centered = init_coords - init_center

        # 快速打分：仅平移到口袋中心，计算 MSE
        quick_coords = centered + pocket_center
        quick_dist = distance_matrix(prot_coords, quick_coords).astype(np.float32)
        quick_dist = np.clip(quick_dist, 0, MAX_DIST)
        quick_score = np.mean((quick_dist - pred_dist) ** 2)
        candidates.append((quick_score, centered))

    if len(candidates) == 0:
        return recover_coords_gradient_descent(pred_dist, prot_coords, n_lig)

    # 按快速打分排序，保留 top-K
    candidates.sort(key=lambda x: x[0])
    top_candidates = candidates[:DE_TOP_K]

    # Step 4: 对 top-K 构象，用 differential_evolution 优化 6+N 自由度
    best_score = float('inf')
    best_coords = None

    for _, centered_coords in top_candidates:

        # 定义打分函数：params -> MSE(当前距离矩阵, 预测距离矩阵)
        def _score_pose(params, _centered=centered_coords):
            tr = np.array(params[:3], dtype=np.float32)          # 平移偏移量
            rot = np.array(params[3:6], dtype=np.float64)        # 轴角旋转
            tor = np.array(params[6:], dtype=np.float64)         # 扭转角

            coords = _centered.copy()

            # (a) 施加扭转角——修改内部自由度
            if n_tor > 0:
                coords = modify_torsion_angles(coords, mol, rot_bonds, tor)

            # (b) 施加旋转——刚体旋转（绕质心）
            rot_mat = axis_angle_to_matrix(rot)
            coords = (rot_mat @ coords.T).T

            # (c) 施加平移——定位到口袋中心 + 偏移
            coords = coords + pocket_center + tr

            # 计算距离矩阵并打分
            current_dist = distance_matrix(prot_coords, coords).astype(np.float32)
            current_dist = np.clip(current_dist, 0, MAX_DIST)
            return np.mean((current_dist - pred_dist) ** 2)

        # 运行 differential_evolution 全局优化
        result = differential_evolution(
            _score_pose, bounds,
            maxiter=DE_MAXITER, popsize=DE_POPSIZE,
            seed=SEED, tol=1e-4, polish=False
        )

        if result.fun < best_score:
            best_score = result.fun
            # 用最优参数重建坐标
            best_params = result.x
            tr = np.array(best_params[:3], dtype=np.float32)
            rot = np.array(best_params[3:6], dtype=np.float64)
            tor = np.array(best_params[6:], dtype=np.float64)
            coords = centered_coords.copy()
            if n_tor > 0:
                coords = modify_torsion_angles(coords, mol, rot_bonds, tor)
            rot_mat = axis_angle_to_matrix(rot)
            coords = (rot_mat @ coords.T).T
            coords = coords + pocket_center + tr
            best_coords = coords.copy()

    # 如果所有构象都因原子数不匹配而跳过，回退到梯度下降
    if best_coords is None:
        return recover_coords_gradient_descent(pred_dist, prot_coords, n_lig)

    return best_coords


# ---- 坐标恢复方法 2: 梯度下降（TankBind 方法，作为对比基线）----

def recover_coords_gradient_descent(pred_dist, prot_coords, n_lig,
                                    n_steps=COORD_RECOVERY_STEPS,
                                    lr=COORD_RECOVERY_LR):
    """从预测距离矩阵恢复配体坐标——使用梯度下降（TankBind 方法）。

    作为构象搜索的对比基线。
    优势：可以精确拟合任意距离矩阵。
    劣势：恢复的坐标可能不满足化学约束（键长/键角可能不合理）。

    参数:
      pred_dist   : (N_p, N_l) numpy 数组
      prot_coords : (N_p, 3)   numpy 数组
      n_lig       : int
      n_steps     : int
      lr          : float

    返回:
      lig_coords  : (N_l, 3) numpy 数组
    """
    with torch.enable_grad():                   # 确保在 no_grad 上下文外也能求梯度
        pred_dist_t = torch.FloatTensor(pred_dist)
        prot_coords_t = torch.FloatTensor(prot_coords)

        center = prot_coords_t.mean(dim=0)
        lig_coords_t = 5.0 * (2.0 * torch.rand(n_lig, 3) - 1.0) + center.unsqueeze(0)
        lig_coords_t = nn.Parameter(lig_coords_t)

        opt = torch.optim.Adam([lig_coords_t], lr=lr)

        for _ in range(n_steps):
            opt.zero_grad()
            current_dist = torch.cdist(prot_coords_t.unsqueeze(0),
                                       lig_coords_t.unsqueeze(0)).squeeze(0)
            current_dist_clamped = torch.clamp(current_dist, max=MAX_DIST)
            loss = ((current_dist_clamped - pred_dist_t) ** 2).mean()
            loss.backward()
            opt.step()

    return lig_coords_t.detach().numpy()


# ---- 实例化模型 ----
model = ToyCarsiDock(atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数量: {total_params:,} (可训练: {trainable_params:,})")
print(f"\n模型结构:\n{model}")

## 5. 训练

训练目标：最小化预测距离矩阵与真实距离矩阵之间的 MSE 损失。

由于每个复合物的原子数不同，使用 `BATCH_SIZE=1` 逐样本处理。每 20 轮在测试集上评估一次验证损失。

In [ ]:
print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for prot_f, lig_f, prot_c, lig_c, dist_mat, idx in train_loader:
        # 每个 batch 只有 1 个样本 (变长结构)，去掉 batch 维度
        prot_f = prot_f.squeeze(0).to(DEVICE)       # (N_p, 10)
        lig_f = lig_f.squeeze(0).to(DEVICE)         # (N_l, 10)
        dist_mat = dist_mat.squeeze(0).to(DEVICE)   # (N_p, N_l)

        optimizer.zero_grad()
        pred_dist = model(prot_f, lig_f)            # (N_p, N_l)
        loss = criterion(pred_dist, dist_mat)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ---- 每 20 轮打印一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_losses = []
        with torch.no_grad():
            for prot_f, lig_f, prot_c, lig_c, dist_mat, idx in test_loader:
                prot_f = prot_f.squeeze(0).to(DEVICE)
                lig_f = lig_f.squeeze(0).to(DEVICE)
                dist_mat = dist_mat.squeeze(0).to(DEVICE)
                pred_dist = model(prot_f, lig_f)
                val_losses.append(criterion(pred_dist, dist_mat).item())

        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_val = np.mean(val_losses) if val_losses else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train Loss: {avg_train:.4f}  |  Val Loss: {avg_val:.4f}")

## 6. 评估与可视化

在测试集上进行以下评估：

1. **距离矩阵预测质量**: 使用 Pearson 相关系数 R 衡量预测距离与真实距离的线性相关性
2. **坐标恢复质量**: 使用 RMSD 衡量恢复坐标与真实坐标的偏差
   - **构象搜索 (CarsiDock)**: 保证化学合理性的 Vina-like 搜索
   - **梯度下降 (TankBind)**: 直接优化坐标的基线方法

评估指标：
- 平均/中位数 RMSD
- RMSD < 2A 的比例（高质量对接）
- RMSD < 5A 的比例（可接受对接）

In [ ]:
print("在测试集上进行预测...")
print("对每个测试样本进行距离矩阵预测 + 两种坐标恢复方法（可能需要几分钟）...")

model.eval()

# 收集指标
all_true_dists = []         # 扁平化的真实距离值
all_pred_dists = []         # 扁平化的预测距离值
rmsds_conformer = []        # 构象搜索方法的 RMSD
rmsds_gradient = []         # 梯度下降方法的 RMSD

with torch.no_grad():
    for i, (prot_f, lig_f, prot_c, lig_c, dist_mat, idx) in enumerate(test_loader):
        prot_f = prot_f.squeeze(0).to(DEVICE)       # (N_p, 10)
        lig_f = lig_f.squeeze(0).to(DEVICE)         # (N_l, 10)
        prot_c = prot_c.squeeze(0).numpy()          # (N_p, 3)
        lig_c = lig_c.squeeze(0).numpy()            # (N_l, 3)
        dist_mat_np = dist_mat.squeeze(0).numpy()   # (N_p, N_l)
        sample_idx = idx.item()

        # 预测距离矩阵
        pred_dist = model(prot_f, lig_f).cpu().numpy()  # (N_p, N_l)

        # 收集距离值用于相关性分析
        all_true_dists.append(dist_mat_np.flatten())
        all_pred_dists.append(pred_dist.flatten())

        n_lig = lig_c.shape[0]

        # 方法 1: 构象搜索（CarsiDock）
        lig_mol = test_dataset.mols[sample_idx]
        coords_conf = recover_coords_conformer_search(
            lig_mol, pred_dist, prot_c, n_confs=N_CONFORMERS
        )
        rmsd_conf = compute_rmsd(coords_conf, lig_c)
        rmsds_conformer.append(rmsd_conf)

        # 方法 2: 梯度下降（TankBind 基线）
        coords_grad = recover_coords_gradient_descent(
            pred_dist, prot_c, n_lig,
            n_steps=COORD_RECOVERY_STEPS, lr=COORD_RECOVERY_LR
        )
        rmsd_grad = compute_rmsd(coords_grad, lig_c)
        rmsds_gradient.append(rmsd_grad)

        if (i + 1) % 10 == 0:
            print(f"  进度: {i + 1}/{len(test_loader)}")

# 合并距离值
all_true_dists = np.concatenate(all_true_dists)
all_pred_dists = np.concatenate(all_pred_dists)
rmsds_conformer = np.array(rmsds_conformer)
rmsds_gradient = np.array(rmsds_gradient)

In [ ]:
# ---- 计算指标 ----
dist_r, _ = pearsonr(all_true_dists, all_pred_dists)

mean_rmsd_conf = np.mean(rmsds_conformer)
median_rmsd_conf = np.median(rmsds_conformer)
pct_under_2_conf = np.mean(rmsds_conformer < 2.0) * 100
pct_under_5_conf = np.mean(rmsds_conformer < 5.0) * 100

mean_rmsd_grad = np.mean(rmsds_gradient)
median_rmsd_grad = np.median(rmsds_gradient)
pct_under_2_grad = np.mean(rmsds_gradient < 2.0) * 100
pct_under_5_grad = np.mean(rmsds_gradient < 5.0) * 100

# ---- 展示结果表格 ----
print(f"测试集样本数: {len(rmsds_conformer)}")
print(f"距离矩阵 Pearson R: {dist_r:.4f}\n")

results_df = pd.DataFrame({
    '指标': ['平均 RMSD (A)', '中位数 RMSD (A)', 'RMSD < 2A (%)', 'RMSD < 5A (%)'],
    '构象搜索 (CarsiDock)': [
        f'{mean_rmsd_conf:.4f}', f'{median_rmsd_conf:.4f}',
        f'{pct_under_2_conf:.1f}', f'{pct_under_5_conf:.1f}'
    ],
    '梯度下降 (TankBind)': [
        f'{mean_rmsd_grad:.4f}', f'{median_rmsd_grad:.4f}',
        f'{pct_under_2_grad:.1f}', f'{pct_under_5_grad:.1f}'
    ]
})
display(results_df)

# ---- 可视化：三张子图 ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# 子图 1: 距离矩阵散点图
ax1 = axes[0]
n_sample = min(5000, len(all_true_dists))
sample_idx = np.random.choice(len(all_true_dists), n_sample, replace=False)
ax1.scatter(all_true_dists[sample_idx], all_pred_dists[sample_idx],
            alpha=0.3, s=8, edgecolors='none', c='steelblue')
lo, hi = 0, MAX_DIST + 1
ax1.plot([lo, hi], [lo, hi], 'r--', linewidth=1.0, label='y = x')
ax1.set_xlabel('True Distance (A)', fontsize=11)
ax1.set_ylabel('Predicted Distance (A)', fontsize=11)
ax1.set_title(f'Distance Matrix  |  R={dist_r:.3f}', fontsize=12)
ax1.legend(loc='upper left')
ax1.set_xlim(lo, hi)
ax1.set_ylim(lo, hi)
ax1.set_aspect('equal')

# 子图 2: 两种方法的 RMSD 直方图对比
ax2 = axes[1]
bins = np.linspace(0, max(rmsds_conformer.max(), rmsds_gradient.max()) + 1, 25)
ax2.hist(rmsds_conformer, bins=bins, color='steelblue', edgecolor='black',
         alpha=0.6, label=f'Conformer Search\n(mean={mean_rmsd_conf:.2f} A)')
ax2.hist(rmsds_gradient, bins=bins, color='coral', edgecolor='black',
         alpha=0.6, label=f'Gradient Descent\n(mean={mean_rmsd_grad:.2f} A)')
ax2.set_xlabel('RMSD (A)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('RMSD Distribution Comparison', fontsize=12)
ax2.legend(loc='upper right', fontsize=9)

# 子图 3: 逐样本 RMSD 对比散点图
ax3 = axes[2]
ax3.scatter(rmsds_gradient, rmsds_conformer, alpha=0.6, s=30,
            edgecolors='k', linewidths=0.3, c='steelblue')
rmsd_max = max(rmsds_conformer.max(), rmsds_gradient.max()) + 1
ax3.plot([0, rmsd_max], [0, rmsd_max], 'r--', linewidth=1.0, label='y = x')
ax3.set_xlabel('Gradient Descent RMSD (A)', fontsize=11)
ax3.set_ylabel('Conformer Search RMSD (A)', fontsize=11)
ax3.set_title('Per-Sample RMSD Comparison', fontsize=12)
ax3.legend(loc='upper left')
ax3.set_xlim(0, rmsd_max)
ax3.set_ylim(0, rmsd_max)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()

## 总结

本教程展示了 CarsiDock 的核心思想——基于 **距离矩阵预测 + 构象搜索** 的分子对接方法。

### 关键要点

1. **网络架构与 TankBind 共享**: 蛋白/配体原子嵌入 -> 外积构建 pair 矩阵 -> 三角更新 -> 距离预测头

2. **核心创新在后处理**: CarsiDock 使用 **RDKit 构象搜索** 替代 TankBind 的梯度下降：
   - 生成多个候选构象（RDKit 力场保证化学合理性）
   - 在 **平移 + 旋转 + 扭转角** 的搜索空间中全局优化（differential_evolution）
   - 变换顺序（扭转 -> 旋转 -> 平移）与 AutoDock Vina 一致

3. **两阶段搜索策略**:
   - **粗筛**: 仅平移的快速打分，选出 top-K 候选
   - **精搜**: 对 top-K 进行 6+N 自由度的 differential_evolution 全局优化

4. **方法对比**:
   - 构象搜索保证键长键角的化学合理性，但搜索空间受限
   - 梯度下降可精确拟合任意距离矩阵，但可能产生化学不合理的构象

### 与原始 CarsiDock 的差异

| 方面 | 本教程 (玩具版) | 原始 CarsiDock |
|------|---------------|---------------|
| 预训练 | 无 | 大规模蛋白-配体预训练 |
| 特征 | 10 维简化特征 | 完整原子/残基特征 |
| 构象搜索 | differential_evolution | 更复杂的搜索策略 |
| 训练数据 | 285 复合物 (CASF-2016) | PDBbind 全集 |
| 三角更新 | 简化版逐元素门控 | AlphaFold2 风格三角乘法 |